# 14 - מצאי רב-אופני והשוואה בין אופני תחבורה

כל אחת מהמחברות הקודמות בפרויקט זה ניתחה רשת **אחת**: גרף שכנות-הנסיעות (trip-adjacency) של כלל ה-GTFS feed הישראלי, כאשר כל אופני התחבורה מאוחדים יחדיו. זהו אובייקט לגיטימי - לנוסע לא משנה איזה סוג כלי רכב מסיע אותו - אך הוא מסתיר עובדה בעלת חשיבות רבה עבור חוסן: ה-feed הוא למעשה **שש רשתות שונות** החולקות מדינה אחת. מקטע אוטובוס ומקטע רכבת הם שניהם "צלע", ואולם עלות ההקמה שלהם שונה, הם כושלים באופנים שונים, ומידת היתירות שלהם שונה לחלוטין.

מחברת זו מפרידה ביניהם. עבור כל `route_type` ב-GTFS היא בונה את גרף שכנות-הנסיעות **הייחודי** של אותו אופן, במעבר streaming יחיד על `stop_times.txt`, ומודדת עבור כולם זה לצד זה את אותם גדלים מבניים: צמתים, צלעות, רכיבי קשירות, נתח הרכיב הגדול ביותר, דרגה ממוצעת, צפיפות, נקודות חיתוך (articulation points) וגשרים (bridges). לאחר מכן היא משווה בין האופנים בטבלה אחת ובשלושה איורים, ומסבירה *מדוע* ההבדלים הם כפי שהם.

**שאלת המחקר.** עד כמה שונים מבנית אופני התחבורה בישראל כאשר כל אחד מהם נבחן כרשת בפני עצמה, ומה משתמע מכך לגבי המקום שבו שוכנת בפועל השבירות של המערכת המשולבת?

מחברת זו היא המסגרת לבלוק המחברות הייעודיות לאופנים שבא אחריה: מחברת 15 (אוטובוסים בלבד), 16 (רכבת קלה והאופנים המשניים) ו-17 (מוקדי מעבר רב-אופניים) - כולן שואבות את היקפן מן המצאי המופק כאן.

## קלט

| נתיב | הופק על ידי | משמש ל |
|---|---|---|
| `outputs/nb/01_data_preparation/tables/routes_clean.csv` | מחברת 01 | `route_id -> route_type` |
| `outputs/nb/01_data_preparation/tables/trips_clean.csv` | מחברת 01 | `trip_id -> route_id` |
| `outputs/nb/01_data_preparation/tables/stops_clean.csv` | מחברת 01 | שמות תחנות, קואורדינטות, מחוז / מטרופולין |
| `outputs/nb/02_graph_construction/tables/nodes.csv`, `edges.csv` | מחברת 02 | בדיקת הצלבה אופציונלית בלבד |
| `israel-public-transportation/stop_times.txt` | ה-feed הגולמי | המקטעים עצמם |

**מחברות שחייבות לרוץ תחילה:** `01_data_preparation` (דרישת חובה). `02_graph_construction` היא אופציונלית - היא משמשת רק להדפסת התאמה (reconciliation), והמחברת מדלגת על בדיקה זו עם הודעה מתאימה אם שלב 02 חסר.

**תלות בנתונים חיצוניים.** `stop_times.txt` שוקל 816 MB / כ-15.7M שורות ו**אינו** מנוהל ב-git. תא ההורדה שלהלן מושך אותו מ-Google Drive בהרצה הראשונה. הקובץ נקרא ב-streaming שורה אחר שורה ולעולם אינו נטען כטבלה שלמה.

## פלט (הכול תחת `outputs/nb/14_multimodal_inventory/`)

| נתיב | תוכן |
|---|---|
| `tables/mode_inventory.csv` | `route_type, mode_label, routes, trips, stops, directed_edges` |
| `tables/mode_network_summary.csv` | `mode_label, nodes, edges, components, largest_component_share, avg_degree, articulation_points, bridges` |
| `tables/mode_structure_details.csv` | אותם אופנים עם צפיפות, דרגה חציונית/מקסימלית, ו*נתחי* נקודות החיתוך והגשרים - העמודות שלא נכנסו לסכמה הנדרשת |
| `tables/edges_by_mode.csv` | `mode_label, from_stop, to_stop, trip_frequency` - תוצאת ה-streaming הגולמית, כך שמחברות האופנים הבאות לא יידרשו לקרוא שוב 816 MB |
| `tables/stop_mode_calls.csv` | `mode_label, stop_id, stop_calls` - עצירות מתוזמנות בכל תחנה **לכל אופן** |
| `tables/stop_modes.csv` | שורה אחת לכל תחנה: אילו אופנים משרתים אותה, כמה, וסך העצירות (קלט נוח למחברת 17, שהיא בעלת האחריות על הניתוח הרב-אופני) |
| `stream_stats.json`, `mode_comparison_summary.json` | מוני ה-streaming והמספרים המרכזיים לכל אופן |
| `figures/mode_size_comparison.png`, `figures/mode_structure_comparison.png`, `figures/mode_map.png` | שלושת האיורים |

דבר מחוץ לתיקייה זו אינו נכתב. התיקיות המצוטטות בדוח - `outputs/tables`, `outputs/figures` ו-`outputs/rail` - אינן נוגעות כלל.

## 1. אתחול סביבת העבודה

זהה לכל שאר המחברות בסדרה, כך שכל הסט רץ באותו אופן מקומית וב-Google Colab. `_ensure(...)` מתקין ב-pip רק חבילות שאכן חסרות, `find_repo_root()` מטפס כלפי מעלה מתיקיית העבודה בחיפוש אחר תיקיית ה-GTFS (ומשכפל את המאגר אם אנו ב-Colab), ולאחר מכן התא קובע את `REPO`, `DATA` ו-`OUT`. כל מה שמופיע להלן תלוי בשלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות השלב, בקרות עלות ואוצר המילים של האופנים

הערימה המדעית (scientific stack) בתוספת מודול הספרייה הסטנדרטית `csv`, שהוא זה שקורא בפועל את ה-feed בנפח 816 MB שורה אחר שורה. שלב זה מחזיק בדיוק תיקייה אחת, `outputs/nb/14_multimodal_inventory/`, ובתוכה `tables/` ו-`figures/`.

**בקרות עלות.** הפעולה היקרה היחידה במחברת זו היא מעבר ה-streaming על `stop_times.txt`: כ-**3-6 דקות** בהתאם למהירות הדיסק, שכן יש לפרסר את כל כ-15.7M השורות. מכיוון שהתוצאה דטרמיניסטית, המעבר כותב את המונים שלו אל `tables/edges_by_mode.csv` ו-`tables/stop_mode_calls.csv`, והרצה מאוחרת יותר טוענת אותם מחדש תוך כשנייה. יש לקבוע `FORCE_RESTREAM = True` כדי להתעלם מן ה-cache ולקרוא מחדש את ה-feed. כל היתר - בניית שישה גרפים, נקודות חיתוך וגשרים - הוא עבודת DFS בזמן לינארי ומסתיים בשניות בודדות אפילו על גרף האוטובוסים בן 26k הצמתים. `MAP_MAX_POINTS_PER_MODE` מגביל כמה נקודות לכל אופן מצוירות באיור הגאוגרפי; רסטריזציה של כ-30k נקודות ב-150 dpi אורכת שניות בודדות.

**תוויות האופנים.** ששת קודי ה-`route_type` הקיימים ב-feed זה מפורטים במפורש. התוויות הועתקו מילה במילה ממחברת 11 כדי ששתי המחברות יסכימו זו עם זו, והן מנוסחות בזהירות מכוונת במקומות שבהם קוד ה-GTFS מטעה: `8` הוא נומינלית "trolleybus" אך ב-feed זה הוא משמש מפעילי מוניות שירות, ו-`715` הוא קוד ה-GTFS ל"שירות אוטובוס לפי דרישה". כל קוד בלתי צפוי מתויג כ-`other (<code>)` במקום להיות מושמט בשקט.

In [ ]:
# --- Libraries, stage folders and cost knobs ------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import csv, json, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

# A handful of rows in stop_times.txt are very long; raise the csv field limit up front.
csv.field_size_limit(10_000_000)

# --- Stage output folders -------------------------------------------------
STAGE = OUT / '14_multimodal_inventory'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Cost knobs (see the markdown above) ---------------------------------
FORCE_RESTREAM = False           # True -> always re-read the 816 MB feed (3-6 minutes)
PROGRESS_EVERY = 2_000_000       # progress print interval during the streaming pass
FIG_DPI = 150                    # figure resolution; drop to 90 for smaller files
MAP_ALPHA = 0.45                 # point transparency on the geographic mode map
MAP_MAX_POINTS_PER_MODE = 40_000 # subsample cap per mode on the map
SEED = 42                        # fixes the map subsample

# --- GTFS mode vocabulary (labels match notebook 11) ----------------------
MODE_LABELS = {
    '0': 'tram/light rail',
    '2': 'rail',
    '3': 'bus',
    '5': 'cable tram',
    '8': 'trolleybus/taxi-coded',
    '715': 'demand/other bus',
}
UNKNOWN_MODE = 'unknown'

def mode_label(route_type):
    """Human-readable name for a GTFS route_type code; never raises, never drops."""
    code = str(route_type).strip()
    if code == '' or code.lower() == 'nan':
        return UNKNOWN_MODE
    return MODE_LABELS.get(code, f'other ({code})')

print('stage folder :', STAGE)
print('modes tracked:', ', '.join(MODE_LABELS.values()))

## 3. רינדור תוויות בעברית

שמות התחנות ב-feed הישראלי הם בעברית, והטבלאות המודפסות להלן מציגות אותם. Matplotlib אינה מממשת את אלגוריתם ה-bidirectional של Unicode, ולכן טקסט מימין לשמאל מצויר הפוך. התא מבצע patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה עברית מומרת לסדר תצוגה באמצעות `python-bidi`, ובוחר גופן הכולל גליפים עבריים. הוא אידמפוטנטי, ולכן הרצה חוזרת אינה מערימה patches נוספים. מכיוון שה-patch גלובלי, יש להעביר ל-matplotlib מכאן ואילך מחרוזות עבריות גולמיות - קריאה ידנית נוספת ל-`fix_he()` תהפוך את הטקסט פעמיים.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור טבלאות ה-GTFS המנוקות משלב 01

שלוש הטבלאות הקטנות הנדרשות למחברת זו מגיעות ממחברת `01_data_preparation`. תיקיות השלבים מאותרות לפי **הקידומת הדו-ספרתית** שלהן ולא לפי slug מדויק, כך שתיקייה ששמה שונה מ-`01_data_preparation` לכל שם אחר המתחיל ב-`01` עדיין תימצא. אם התיקייה או הקובץ חסרים, פונקציות העזר זורקות `FileNotFoundError` המציין את שם המחברת שיש להריץ - נפילה שקטה לברירת מחדל כאן הייתה מייצרת מצאי אופנים שבו כל נסיעה מתויגת `unknown`, מצב שנראה סביר אך שגוי לחלוטין.

הכול נקרא כ**מחרוזות**. מזהי GTFS הם קודים אטומים; מתן אפשרות ל-pandas להסיק טיפוסים היה הופך את `route_type` למספר שלם, מסיר אפסים מובילים ממזהים, ושובר את ה-joins שלהלן.

In [ ]:
# --- Resolve earlier stages by their two-digit prefix ---------------------
def find_stage(prefix, notebook_hint):
    """Return an earlier stage's output folder, matched by its NN prefix."""
    matches = sorted(p for p in OUT.glob(f'{prefix}*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f'No stage folder starting with "{prefix}" under {OUT}. '
            f'Run notebook {notebook_hint} first.')
    return matches[0]


def stage_artifact(prefix, filename, notebook_hint):
    """Path of `filename` inside stage `prefix`, or a FileNotFoundError that says why."""
    stage = find_stage(prefix, notebook_hint)
    direct = stage / 'tables' / filename
    if direct.exists():
        return direct
    hits = sorted(stage.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f'{filename} not found under {stage}. '
            f'Run notebook {notebook_hint} first - it writes {filename}.')
    return hits[0]


def read_gtfs_table(path):
    """GTFS ids are opaque codes: read every column as a string, keep blanks as ''."""
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


routes = read_gtfs_table(stage_artifact('01', 'routes_clean.csv', '01_data_preparation'))
trips = read_gtfs_table(stage_artifact('01', 'trips_clean.csv', '01_data_preparation'))
stops = read_gtfs_table(stage_artifact('01', 'stops_clean.csv', '01_data_preparation'))

for name, frame, column in [('routes_clean', routes, 'route_type'),
                            ('trips_clean', trips, 'route_id'),
                            ('stops_clean', stops, 'stop_id')]:
    if column not in frame.columns:
        raise KeyError(f'{name}.csv has no "{column}" column - re-run 01_data_preparation.')

print(f'routes: {len(routes):,}   trips: {len(trips):,}   stops: {len(stops):,}')
print('route_type codes present:', sorted(set(routes['route_type'])))

## 5. מסלולים ונסיעות מתוזמנות לפי אופן, ומיפוי `trip_id -> mode`

האופן מצוי ב-`routes.txt`, אך הקובץ שעלינו לקרוא ב-streaming (`stop_times.txt`) מכיר רק `trip_id`. הגשר הוא `trips.txt`, ולכן אנו מרכיבים את שני ה-lookups פעם אחת, בזיכרון, לכדי מילון יחיד `trip_id -> mode_label` (כ-420k רשומות - עשרות MB בודדות, זניח לעומת 816 MB שאנו עומדים לקרוא). כל מקטע הנמצא במהלך מעבר ה-streaming מנותב לאופן שלו באמצעות חיפוש יחיד במילון.

אותו join מספק את המחצית הראשונה של `mode_inventory.csv` הנדרש: כמה **רשומות מסלול** וכמה **נסיעות מתוזמנות** תורם כל אופן. יש לשים לב ש"רשומת מסלול" ב-GTFS היא כיוון-ווריאנט של קו, ולא קו - ומכאן 962 "מסלולי" רכבת עבור רשת של כ-67 תחנות. נסיעות שה-`route_id` שלהן אינו מופיע ב-`routes_clean.csv` היו הופכות בשקט ל-`unknown`, ולכן אנו סופרים אותן במפורש.

In [ ]:
# --- route_id -> route_type -> mode label, then trip_id -> mode label -----
route_type_of = dict(zip(routes['route_id'], routes['route_type']))
trip_mode = {trip: mode_label(route_type_of.get(route, ''))
             for trip, route in zip(trips['trip_id'], trips['route_id'])}

orphan_trips = sum(1 for route in trips['route_id'] if route not in route_type_of)

# Route records and scheduled trips per mode, straight from the small GTFS tables.
trips_typed = trips.assign(route_type=trips['route_id'].map(route_type_of).fillna(''))
meta = pd.DataFrame({
    'routes': routes.groupby('route_type').size(),
    'trips': trips_typed.groupby('route_type').size(),
}).fillna(0).astype(int)
meta.index.name = 'route_type'
meta = meta.reset_index()
meta['mode_label'] = meta['route_type'].map(mode_label)
meta = meta.sort_values('trips', ascending=False).reset_index(drop=True)

print(f'trip_id -> mode entries: {len(trip_mode):,}')
print(f'trips whose route_id is missing from routes_clean.csv: {orphan_trips:,}')
meta

## 6. תלות בנתונים חיצוניים: `stop_times.txt`

`stop_times.txt` שוקל 816 MB - הרבה מעל מגבלת גודל הקובץ של GitHub - ולכן הוא **אינו** במאגר. התא שלהלן מוריד אותו מ-Google Drive בהרצה הראשונה ומדלג על ההורדה אם הקובץ כבר קיים. זוהי תלות הרשת החיצונית היחידה של המחברת; כל היתר נמצא במאגר או מופק על ידי מחברת 01. ההורדה אורכת דקות בודדות בהרצת Colab ראשונה.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 7. מעבר streaming אחד, מונה צלעות אחד לכל אופן

זהו התא היקר: **3-6 דקות** עבור כ-15.7M שורות. הדרך הנאיבית להשוות שישה אופנים הייתה שישה מעברים על הקובץ (או מעבר אחד לכל אופן, כפי שעושה מחברת 11 עבור הרכבת בלבד). במקום זאת אנו מבצעים מעבר **אחד** ומנתבים כל מקטע למונה של האופן שלו תוך כדי תנועה, מה שעולה חיפוש מילון נוסף אחד לכל נסיעה והופך משימה של חצי שעה למשימה של חמש דקות.

המצב הנשמר לכל שורה הוא O(1); המצב הנשמר בסך הכול הוא O(מספר המקטעים הייחודיים), כ-52k רשומות בסך הכול על פני כל האופנים. עבור כל אופן אנו צוברים:

* `edge_counts[mode][(u, v)]` - מספר הנסיעות של אותו אופן העוברות במקטע `u -> v`;
* `stop_calls[mode][stop]` - עצירות מתוזמנות של אותו אופן באותה תחנה (המקבילה לכל אופן של `stop_use_count` בשלב 02);
* `trips_observed[mode]`, `rows_by_mode[mode]` - מוני דיווח.

**הנחת הסדר.** בדיוק כמו במחברת 02, המעבר מניח כי `stop_times.txt` ממוין לפי `(trip_id, stop_sequence)`, כך ששורות עוקבות של אותה נסיעה הן תחנות עוקבות. מחברת 02 בדקה הנחה זו על פני הקובץ המלא ומצאה אפס נסיגות ב-`stop_sequence` ואפס בלוקים משורגים של נסיעות, ולכן ההנחה מנוצלת כאן מחדש במקום להיבדק שוב; מאותה סיבה גם חיפוש האופן מתבצע פעם אחת לכל בלוק נסיעה. אילו בדיקה זו הייתה נכשלת אי פעם, הצלעות הנבנות כאן היו שגויות באותו אופן שבו היו שגויות אלו של שלב 02.

יש לשים לב שלא מתבצע כאן כלל פרסור זמנים, ולכן מלכודת ה-GTFS של "שעות >= 24" (`25:30:00` פירושו 01:30 ביום השירות הבא) אינה מתעוררת כאן כלל - זו בעייתה של מחברת 18. נסיעות שאינן מופיעות ב-`trips_clean.csv` נספרות תחת `unknown` במקום להיות מושמטות.

**Caching.** התוצאה דטרמיניסטית, ולכן לאחר ההרצה הראשונה היא נכתבת אל `tables/edges_by_mode.csv` ו-`tables/stop_mode_calls.csv` ונטענת משם מחדש תוך כשנייה. `FORCE_RESTREAM = True` עוקף את ה-cache.

In [ ]:
# --- One pass over stop_times.txt, routing every segment to its mode ------
EDGES_BY_MODE = TABLES / 'edges_by_mode.csv'
STOP_MODE_CALLS = TABLES / 'stop_mode_calls.csv'
STREAM_STATS = STAGE / 'stream_stats.json'


def stream_mode_edges(path, trip_mode, progress_every=PROGRESS_EVERY):
    """Build one segment-weight counter per transport mode in a single streaming pass.

    Assumes the feed is sorted by (trip_id, stop_sequence), verified in notebook 02.
    Memory is O(|E| + |V|), never O(rows).
    """
    edge_counts = defaultdict(Counter)   # mode -> Counter[(u, v)] = trips on that segment
    stop_calls = defaultdict(Counter)    # mode -> Counter[stop_id] = scheduled stop calls
    trips_observed = Counter()           # mode -> distinct trips met in the feed
    rows_by_mode = Counter()
    rows_read = 0
    self_loops = 0
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        for field in ('trip_id', 'stop_id'):
            if field not in header:
                raise ValueError(f'stop_times.txt has no "{field}" column')
        i_trip, i_stop = header.index('trip_id'), header.index('stop_id')

        prev_trip, prev_stop, mode = None, None, UNKNOWN_MODE
        for row in reader:
            rows_read += 1
            trip, stop = row[i_trip], row[i_stop]

            if trip != prev_trip:
                # New trip block: resolve the mode once per trip, not once per row,
                # and reset the predecessor so no edge crosses a trip boundary.
                mode = trip_mode.get(trip, UNKNOWN_MODE)
                trips_observed[mode] += 1
                prev_stop = None

            rows_by_mode[mode] += 1
            stop_calls[mode][stop] += 1
            if prev_stop is not None:
                if prev_stop != stop:
                    edge_counts[mode][(prev_stop, stop)] += 1
                else:
                    self_loops += 1

            prev_trip, prev_stop = trip, stop
            if progress_every and rows_read % progress_every == 0:
                print(f'    {rows_read:,} rows | {time.time() - t0:,.0f}s')

    stats = {
        'rows_read': rows_read,
        'self_loop_rows_skipped': self_loops,
        'rows_by_mode': dict(rows_by_mode),
        'trips_observed': dict(trips_observed),
        'elapsed_seconds': round(time.time() - t0, 1),
    }
    return edge_counts, stop_calls, stats


cache_ready = all(p.exists() for p in (EDGES_BY_MODE, STOP_MODE_CALLS, STREAM_STATS))
if cache_ready and not FORCE_RESTREAM:
    print('Re-using the cached per-mode counters from an earlier run of this notebook.')
    _cached_edges = pd.read_csv(EDGES_BY_MODE, dtype=str, keep_default_na=False,
                                encoding='utf-8-sig')
    edge_counts = defaultdict(Counter)
    for m, u, v, w in zip(_cached_edges['mode_label'], _cached_edges['from_stop'],
                          _cached_edges['to_stop'], _cached_edges['trip_frequency']):
        edge_counts[m][(u, v)] = int(w)
    _cached_calls = pd.read_csv(STOP_MODE_CALLS, dtype=str, keep_default_na=False,
                                encoding='utf-8-sig')
    stop_calls = defaultdict(Counter)
    for m, s, c in zip(_cached_calls['mode_label'], _cached_calls['stop_id'],
                       _cached_calls['stop_calls']):
        stop_calls[m][s] = int(c)
    with open(STREAM_STATS, encoding='utf-8') as handle:
        stream_stats = json.load(handle)
else:
    print('Streaming stop_times.txt (~15.7M rows) - this takes a few minutes ...')
    edge_counts, stop_calls, stream_stats = stream_mode_edges(STOP_TIMES, trip_mode)

print('rows read       : {:,}'.format(stream_stats['rows_read']))
print('self-loop rows  : {:,}'.format(stream_stats['self_loop_rows_skipped']))
print('elapsed seconds :', stream_stats['elapsed_seconds'])
print()
print('{:<24}{:>16}{:>10}{:>9}{:>11}'.format('mode', 'stop-time rows', 'trips',
                                             'stops', 'segments'))
for m, n_rows in sorted(stream_stats['rows_by_mode'].items(), key=lambda kv: -kv[1]):
    print('{:<24}{:>16,}{:>10,}{:>9,}{:>11,}'.format(
        m, n_rows, stream_stats['trips_observed'].get(m, 0),
        len(stop_calls.get(m, {})), len(edge_counts.get(m, {}))))

## 8. שמירת תוצאת ה-streaming

שתי טבלאות בפורמט long נכתבות מיד, לפני כל ניתוח, משתי סיבות: הן ה-cache שהופך הרצה חוזרת של מחברת זו לזולה, והן ההעברה למחברות הייעודיות לאופנים הבאות אחריה (15 אוטובוס, 16 רכבת קלה ואופנים משניים, 17 מוקדים רב-אופניים) כך שאף אחת מהן לא תידרש לגעת שוב ב-feed בנפח 816 MB.

* `tables/edges_by_mode.csv` - `mode_label, from_stop, to_stop, trip_frequency`, שורה אחת לכל (אופן, מקטע מכוון).
* `tables/stop_mode_calls.csv` - `mode_label, stop_id, stop_calls`, שורה אחת לכל (אופן, תחנה).
* `stream_stats.json` - מוני השורות / הנסיעות, כך שלמספרים המצוטטים בדוח יש מקור קריא-מכונה.

כל קובצי ה-CSV נכתבים כ-UTF-8 עם BOM כדי שעברית תיפתח כראוי ב-Excel.

In [ ]:
# --- Persist the streaming result (also serves as the cache) --------------
edges_by_mode = pd.DataFrame(
    [{'mode_label': m, 'from_stop': u, 'to_stop': v, 'trip_frequency': int(w)}
     for m, counter in edge_counts.items() for (u, v), w in counter.items()]
)
edges_by_mode.to_csv(EDGES_BY_MODE, index=False, encoding='utf-8-sig')

stop_mode_calls = pd.DataFrame(
    [{'mode_label': m, 'stop_id': s, 'stop_calls': int(c)}
     for m, counter in stop_calls.items() for s, c in counter.items()]
)
stop_mode_calls.to_csv(STOP_MODE_CALLS, index=False, encoding='utf-8-sig')

with open(STREAM_STATS, 'w', encoding='utf-8') as handle:
    json.dump(stream_stats, handle, ensure_ascii=False, indent=2)

print(f'{len(edges_by_mode):,} (mode, segment) rows -> {EDGES_BY_MODE}')
print(f'{len(stop_mode_calls):,} (mode, stop) rows    -> {STOP_MODE_CALLS}')
print(f'streaming counters                  -> {STREAM_STATS}')
edges_by_mode.head()

## 9. גרף אחד לכל אופן, נמדד באותו אופן

כל אופן מקבל כעת גרף שכנות-נסיעות משלו, הבנוי בדיוק לפי ההגדרה ששימשה עבור הרשת השלמה במחברת 02: צומת = תחנה המשורתת על ידי אותו אופן, צלע מכוונת `u -> v` = נסיעה כלשהי של אותו אופן עוצרת ב-`v` מיד לאחר `u`, משקל = מספר הנסיעות מסוג זה. ההטלה הלא-מכוונת **מסכמת** את משקלי שני הכיוונים, משום שתחנה סגורה או מקטע חסום עוצרים תנועה בשני הכיוונים.

עבור כל אופן אנו מחשבים לאחר מכן, הכול בשגרות בזמן לינארי:

* **גודל** - צמתים, צלעות לא מכוונות, צלעות מכוונות;
* **דלילות** - צפיפות `2m / (n(n-1))`, דרגה ממוצעת / חציונית / מקסימלית;
* **פיצול** - מספר רכיבי הקשירות ונתח התחנות ברכיב הגדול ביותר;
* **מבנה חתכים** - נקודות חיתוך (articulation points; צמתים שהסרתם מפרקת את הגרף) וגשרים (bridges; צלעות בעלות אותה תכונה), בתוספת כל אחד מהם כ*נתח* מן הצמתים / הצלעות, שהיא הדרך ההוגנת היחידה להשוות רשת אוטובוסים בת 26,000 צמתים לרשת רכבת בת 67 תחנות.

אופנים המופיעים ב-`routes_clean.csv` אך לא הפיקו ולו מקטע אחד עדיין מקבלים שורת אפסים במקום להיעלם מן ההשוואה. התא כולו רץ בשניות בודדות; גרף האוטובוסים שולט בזמן הריצה.

In [ ]:
# --- Build and measure one network per mode ------------------------------
def build_mode_graphs(counter):
    """Directed trip-adjacency graph for one mode, plus its summed undirected view."""
    D = nx.DiGraph()
    for (u, v), w in counter.items():
        D.add_edge(u, v, weight=int(w))
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        if G.has_edge(u, v):
            G[u][v]['weight'] += data['weight']
        else:
            G.add_edge(u, v, weight=data['weight'])
    return G, D


def summarise_mode(label, G, D):
    """Size, sparsity, fragmentation and cut structure of a single mode's network."""
    n = G.number_of_nodes()
    m = G.number_of_edges()
    sizes = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    degrees = np.array([d for _, d in G.degree()]) if n else np.array([0])
    articulation = sorted(set(nx.articulation_points(G))) if n else []
    bridges = list(nx.bridges(G)) if n else []
    return {
        'mode_label': label,
        'nodes': n,
        'edges': m,
        'directed_edges': D.number_of_edges(),
        'components': len(sizes),
        'largest_component_nodes': sizes[0] if sizes else 0,
        'largest_component_share': round(sizes[0] / n, 4) if n else 0.0,
        'avg_degree': round(float(degrees.mean()), 2) if n else 0.0,
        'median_degree': float(np.median(degrees)) if n else 0.0,
        'max_degree': int(degrees.max()) if n else 0,
        'density': round(nx.density(G), 6) if n > 1 else 0.0,
        'articulation_points': len(articulation),
        'bridges': len(bridges),
        'articulation_share': round(len(articulation) / n, 4) if n else 0.0,
        'bridge_share': round(len(bridges) / m, 4) if m else 0.0,
    }


observed_modes = sorted(edge_counts, key=lambda m: -len(edge_counts[m]))
declared_only = [m for m in meta['mode_label'] if m not in observed_modes]

mode_graphs = {}
rows = []
for label in observed_modes + declared_only:
    t0 = time.time()
    G_mode, D_mode = build_mode_graphs(edge_counts.get(label, Counter()))
    mode_graphs[label] = (G_mode, D_mode)
    rows.append(summarise_mode(label, G_mode, D_mode))
    print('{:<24}{:>8,} nodes{:>9,} edges   ({:.1f}s)'.format(
        label, G_mode.number_of_nodes(), G_mode.number_of_edges(), time.time() - t0))

structure = pd.DataFrame(rows).sort_values('nodes', ascending=False).reset_index(drop=True)
structure

## 10. `tables/mode_inventory.csv`

הפלט הנדרש הראשון: שורה אחת לכל אופן עם ארבעת נפחי הכותרת - `routes` ו-`trips` ממטא-הנתונים של GTFS, `stops` ו-`directed_edges` כפי שנצפו ב-feed עצמו. שתי המחציות עונות על שאלות שונות. `routes` ו-`trips` אומרים כמה **שירות** מפעיל אופן מסוים; `stops` ו-`directed_edges` אומרים כמה **רשת** הוא מכסה. רכבת ורכבת קלה הן ההמחשה הברורה ביותר: הרכבת הקלה מפעילה יותר מפי שניים נסיעות מן הרכבת הכבדה על פני רשת קטנה בסדר גודל שלם.

`trips` הוא מספר רשומות הנסיעה המתוזמנות ב-`trips.txt`, ולא נסיעות ליום - ה-feed הוא תצלום מצב של חלון לוח זמנים אחד, ורשומת נסיעה בודדת היא הרצה מתוזמנת אחת. כל אופן המופיע ב-`stop_times.txt` אך לא ב-`routes_clean.csv` (כלומר `unknown`) מצורף כשורה מפורשת במקום להיות מושמט, כך שהטבלה תמיד נותנת דין וחשבון על כל שורת stop-time.

In [ ]:
# --- tables/mode_inventory.csv -------------------------------------------
observed_stops = {m: len(c) for m, c in stop_calls.items()}
observed_edges = {m: len(c) for m, c in edge_counts.items()}

inventory = meta.copy()
inventory['stops'] = inventory['mode_label'].map(observed_stops).fillna(0).astype(int)
inventory['directed_edges'] = (inventory['mode_label'].map(observed_edges)
                               .fillna(0).astype(int))

# Modes seen while streaming but absent from routes_clean.csv are added, not dropped.
extra_modes = [m for m in observed_stops if m not in set(inventory['mode_label'])]
if extra_modes:
    print('Modes present in stop_times.txt but not in routes_clean.csv:', extra_modes)
    inventory = pd.concat([inventory, pd.DataFrame([
        {'route_type': '', 'mode_label': m, 'routes': 0, 'trips': 0,
         'stops': observed_stops[m], 'directed_edges': observed_edges.get(m, 0)}
        for m in extra_modes])], ignore_index=True)

mode_inventory = (inventory[['route_type', 'mode_label', 'routes', 'trips',
                             'stops', 'directed_edges']]
                  .sort_values('trips', ascending=False)
                  .reset_index(drop=True))
mode_inventory.to_csv(TABLES / 'mode_inventory.csv', index=False, encoding='utf-8-sig')

print('saved:', TABLES / 'mode_inventory.csv')
mode_inventory

## 11. `tables/mode_network_summary.csv` וטבלת הפירוט המורחבת

הפלט הנדרש השני מחזיק את ההשוואה המבנית בדיוק בסכמה שהמחברות שבהמשך מצפות לה: `mode_label, nodes, edges, components, largest_component_share, avg_degree, articulation_points, bridges`. כדי לשמור על חוזה זה נקי, הגדלים הנוספים שנמדדו לעיל - צפיפות, דרגה חציונית ומקסימלית, מספר הצלעות המכוונות, גודל הרכיב הגדול ביותר, ו**נתחי** נקודות החיתוך / הגשרים שהופכים אופנים קטנים וגדולים לברי-השוואה - עוברים לטבלה נלווית `tables/mode_structure_details.csv`, יחד עם נפחי המסלולים והנסיעות, כך שקובץ יחיד תומך בכל הדיון.

`mode_comparison_summary.json` נושא את אותם מספרים בצורה מקוננת עבור הדוח.

In [ ]:
# --- tables/mode_network_summary.csv (exact required schema) -------------
mode_network_summary = structure[[
    'mode_label', 'nodes', 'edges', 'components', 'largest_component_share',
    'avg_degree', 'articulation_points', 'bridges']].copy()
mode_network_summary.to_csv(TABLES / 'mode_network_summary.csv',
                            index=False, encoding='utf-8-sig')

# --- tables/mode_structure_details.csv (everything else we measured) -----
details = structure.merge(
    mode_inventory[['mode_label', 'route_type', 'routes', 'trips']],
    on='mode_label', how='left')
details['routes'] = details['routes'].fillna(0).astype(int)
details['trips'] = details['trips'].fillna(0).astype(int)
details['route_type'] = details['route_type'].fillna('')
details['observed_trips'] = (details['mode_label']
                             .map(stream_stats['trips_observed']).fillna(0).astype(int))
details['stop_time_rows'] = (details['mode_label']
                             .map(stream_stats['rows_by_mode']).fillna(0).astype(int))
details['trips_per_edge'] = np.where(
    details['edges'] > 0,
    (details['observed_trips'] / details['edges'].replace(0, np.nan)).round(2), 0.0)
details = details[[
    'route_type', 'mode_label', 'routes', 'trips', 'observed_trips', 'stop_time_rows',
    'nodes', 'edges', 'directed_edges', 'density', 'avg_degree', 'median_degree',
    'max_degree', 'components', 'largest_component_nodes', 'largest_component_share',
    'articulation_points', 'articulation_share', 'bridges', 'bridge_share',
    'trips_per_edge']]
details.to_csv(TABLES / 'mode_structure_details.csv', index=False, encoding='utf-8-sig')

# --- mode_comparison_summary.json ---------------------------------------
comparison_summary = {
    'modes': int(len(structure)),
    'total_stop_time_rows': int(stream_stats['rows_read']),
    'total_directed_segments_all_modes': int(structure['directed_edges'].sum()),
    'per_mode': {row['mode_label']: {key: row[key] for key in (
        'nodes', 'edges', 'directed_edges', 'components', 'largest_component_share',
        'avg_degree', 'density', 'articulation_points', 'articulation_share',
        'bridges', 'bridge_share')} for _, row in structure.iterrows()},
}
with open(STAGE / 'mode_comparison_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(comparison_summary, handle, ensure_ascii=False, indent=2, default=float)

print('saved:', TABLES / 'mode_network_summary.csv')
print('saved:', TABLES / 'mode_structure_details.csv')
print('saved:', STAGE / 'mode_comparison_summary.json')
details

## 12. בדיקת הצלבה: האם האופנים מסתכמים לגרף הרשת השלמה?

פיצול מסוג זה קל לבצע באופן שגוי בעדינות - join לקוי היה שולח בשקט מחצית מנסיעות האוטובוס ל-`unknown` וכל המספרים לעיל עדיין היו נראים סבירים. לכן אנו מבצעים התאמה מול מחברת 02, שבנתה את אותו גרף ללא כל פיצול לפי אופן:

* **האיחוד** של קבוצות התחנות לפי אופן אמור להתאים לתחנות הפעילות של שלב 02 (קובץ `nodes.csv` של שלב 02 קטן במקצת משום שתחנה ללא קודמת וללא עוקבת אינה צומת שם, בעוד שהיא כן מופיעה במוני עצירות התחנה שלנו);
* **האיחוד** של קבוצות המקטעים לפי אופן אמור להתאים כמעט בדיוק למספר הצלעות המכוונות של שלב 02 - השניים אמורים להיות זהים, משום ששניהם סופרים זוגות `(from_stop, to_stop)` ייחודיים על פני אותו קובץ;
* **הסכום** על פני האופנים של מספרי התחנות גדול מן האיחוד הזה, והפער הוא בדיוק התחנות הרב-אופניות שמחברת 17 ממשיכה וחוקרת. אותו דבר נכון לגבי מקטעים המשורתים על ידי יותר מאופן אחד.

הבדיקה אופציונלית: אם שלב 02 לא הורץ, התא מדפיס מדוע הוא מדלג במקום להיכשל.

In [ ]:
# --- Reconcile the per-mode split against the whole-network graph --------
union_stops = set().union(*(set(c) for c in stop_calls.values())) if stop_calls else set()
union_edges = set().union(*(set(c) for c in edge_counts.values())) if edge_counts else set()
sum_stops = sum(len(c) for c in stop_calls.values())
sum_edges = sum(len(c) for c in edge_counts.values())

print(f'union of per-mode stops    : {len(union_stops):,}')
print(f'sum  of per-mode stops     : {sum_stops:,}   (a stop counted once per mode)')
print(f'union of per-mode segments : {len(union_edges):,}')
print(f'sum  of per-mode segments  : {sum_edges:,}')
print(f'multimodal surplus (stops) : {sum_stops - len(union_stops):,} extra (mode, stop) pairs')

try:
    nodes_all = pd.read_csv(stage_artifact('02', 'nodes.csv', '02_graph_construction'),
                            dtype=str, keep_default_na=False, encoding='utf-8-sig')
    edges_all = pd.read_csv(stage_artifact('02', 'edges.csv', '02_graph_construction'),
                            dtype=str, keep_default_na=False, encoding='utf-8-sig')
    print()
    print(f'stage 02 nodes.csv rows    : {len(nodes_all):,}')
    print(f'stage 02 edges.csv rows    : {len(edges_all):,}  (directed)')
    print(f'segment difference         : {len(union_edges) - len(edges_all):+,}')
except FileNotFoundError as err:
    print()
    print('Skipping the stage-02 cross-check:', err)

## 13. מבט ברמת התחנה: אילו אופנים משרתים כל תחנה

הטבלה הארוכה `stop_mode_calls` עוברת pivot לשורה אחת לכל תחנה, ומצורפת לשמות ולקואורדינטות מתוך `stops_clean.csv`. זהו המקור שממנו מצויר המפה שלהלן, והוא נכתב אל `tables/stop_modes.csv` כקלט נוח למחברת 17 - **מחברת 17 היא בעלת האחריות על ניתוח המוקדים הרב-אופניים** (`transfer_hubs.csv`, הסכמה והספים משלה); טבלה זו היא רק העובדה הגולמית "אילו אופנים נוגעים בתחנה זו", המחושבת כאן משום שמעבר ה-streaming הוא המקום שבו זול להשיגה.

`modes_served` הוא רשימה מופרדת ב-`|` וממוינת אלפביתית, כך שהיא יציבה ונוחה ל-diff. תחנות המופיעות ב-feed אך חסרות מ-`stops_clean.csv` (הושמטו במהלך הניקוי, למשל בשל קואורדינטות לא תקינות) שומרות על שורתן עם תכונות ריקות, ומספרן מדווח ולא מוסתר.

In [ ]:
# --- One row per stop: which modes serve it ------------------------------
geo = stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].copy()
geo['lat'] = pd.to_numeric(geo['stop_lat'], errors='coerce')
geo['lon'] = pd.to_numeric(geo['stop_lon'], errors='coerce')
for column in ('region', 'metro'):
    geo[column] = stops[column] if column in stops.columns else ''
geo = geo[['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro']]

stop_mode_geo = stop_mode_calls.merge(geo, on='stop_id', how='left')
missing_attrs = int(stop_mode_geo['stop_name'].isna().sum())
missing_coords = int(stop_mode_geo['lat'].isna().sum())

stop_modes = (stop_mode_geo
              .groupby('stop_id', as_index=False)
              .agg(stop_name=('stop_name', 'first'),
                   lat=('lat', 'first'),
                   lon=('lon', 'first'),
                   region=('region', 'first'),
                   metro=('metro', 'first'),
                   n_modes=('mode_label', 'nunique'),
                   modes_served=('mode_label', lambda s: '|'.join(sorted(set(s)))),
                   stop_calls=('stop_calls', 'sum')))
stop_modes = stop_modes.sort_values(['n_modes', 'stop_calls'], ascending=False)
stop_modes.to_csv(TABLES / 'stop_modes.csv', index=False, encoding='utf-8-sig')

print(f'stops in the feed                  : {len(stop_modes):,}')
print(f'  serving more than one mode       : {int((stop_modes["n_modes"] > 1).sum()):,}')
print(f'  (mode, stop) rows with no name   : {missing_attrs:,}')
print(f'  (mode, stop) rows with no coords : {missing_coords:,}')
print('saved:', TABLES / 'stop_modes.csv')
stop_modes.head(10)

## 14. איור 1 - מה גודלו של כל אופן?

תרשים עמודות מקובץ של שלושת מדדי הגודל לכל אופן: תחנות משורתות, מקטעים לא מכוונים, ונסיעות מתוזמנות. ציר ה-y הוא **לוגריתמי**, וזו אינה בחירה קוסמטית: האוטובוס גדול בשלושה עד ארבעה סדרי גודל מן האופנים הקטנים ביותר בכל ציר, כך שבסקאלה לינארית חמש מתוך שש העמודות היו בלתי נראות. הסקאלה הלוגריתמית היא שהופכת את ההשוואה לקריאה, והיא גם הדרך הישרה להציג את הטענה: אלו אינן שש רשתות ברות-השוואה, אלא רשת ארצית אחת בתוספת חמש רשתות קטנות ומתמחות.

יש לשים לב להצטלבויות. עבור אוטובוס, גם הנסיעות וגם התחנות עצומות. עבור cable tram ורכבת קלה, עמודת הנסיעות מתנשאת מעל עמודות התחנות והמקטעים - שירות רב מרוכז על גבי רשת זעירה. יחס זה מכומת כ-`trips_per_edge` בטבלת הפירוט.

In [ ]:
# --- Figure 1: grouped bar chart of size metrics per mode ----------------
plot_df = (mode_network_summary
           .merge(mode_inventory[['mode_label', 'trips']], on='mode_label', how='left')
           .fillna({'trips': 0})
           .sort_values('nodes', ascending=False)
           .reset_index(drop=True))

metrics = [('stops (nodes)', 'nodes'),
           ('segments (undirected edges)', 'edges'),
           ('scheduled trips', 'trips')]
x = np.arange(len(plot_df))
width = 0.26
colors = ['#3b6ea5', '#e07b39', '#5a9e6f']
ceiling = float(plot_df[['nodes', 'edges', 'trips']].to_numpy().max())

fig, ax = plt.subplots(figsize=(11, 6))
for k, (label, column) in enumerate(metrics):
    values = [max(int(v), 0) for v in plot_df[column]]
    bars = ax.bar(x + (k - 1) * width, values, width, label=label, color=colors[k])
    ax.bar_label(bars, labels=[f'{v:,}' for v in values],
                 fontsize=7, padding=2, rotation=90)

ax.set_yscale('log')
ax.set_ylim(0.7, ceiling * 12 if ceiling > 0 else 10)
ax.set_xticks(x)
ax.set_xticklabels(plot_df['mode_label'], rotation=20, ha='right')
ax.set_ylabel('count (log scale)')
ax.set_title('Size of each transport mode as its own network\n'
             '(log scale - bus dwarfs every other mode on all three axes)')
ax.legend(loc='upper right', frameon=True)
fig.tight_layout()
fig.savefig(FIGURES / 'mode_size_comparison.png', dpi=FIG_DPI)
plt.show()

print('saved:', FIGURES / 'mode_size_comparison.png')

## 15. איור 2 - עד כמה שביר כל אופן?

ספירות מוחלטות של נקודות חיתוך אינן ניתנות להשוואה בין אופנים: לאוטובוס יש אלפים מהן פשוט משום שיש לו עשרות אלפי צמתים. שלושת הפאנלים שלהלן משתמשים לפיכך בגדלים **מנורמלים**:

1. **דרגה ממוצעת** - בכמה תחנות שכנות מחוברת התחנה הטיפוסית. ערך קרוב ל-2 פירושו שהאופן הוא בעצם אוסף של מסלולים; מעל 3 פירושו רישות (meshing) של ממש.
2. **נתח נקודות החיתוך** - שיעור התחנות של אותו אופן שסגירתן תפצל את רשתו. זהו מספר השבירות הישיר ביותר במחברת זו.
3. **נתח הרכיב הגדול ביותר** - כמה מן האופן מהווה מערכת קשירה אחת ולא איים מנותקים.

יש לקרוא אותם יחד: אופן יכול להיראות חסין באחד מהם וגרוע באחר. אופן המורכב מהסעות רבות ונפרדות בנות שתי תחנות יבחין בנתח נקודות חיתוך נמוך (אין מה לחתוך) *וגם* בנתח רכיב גדול ביותר גרוע.

In [ ]:
# --- Figure 2: normalised structure / fragility per mode -----------------
frag = details.sort_values('nodes', ascending=False).reset_index(drop=True)
panels = [('avg_degree', 'Average degree', '#3b6ea5'),
          ('articulation_share', 'Share of stations that are cut vertices', '#c0504d'),
          ('largest_component_share', 'Share of stations in the largest component',
           '#5a9e6f')]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (column, title, color) in zip(axes, panels):
    values = [float(v) for v in frag[column]]
    bars = ax.bar(range(len(frag)), values, color=color)
    ax.bar_label(bars, labels=[f'{v:.2f}' for v in values], fontsize=8, padding=2)
    ax.set_xticks(range(len(frag)))
    ax.set_xticklabels(frag['mode_label'], rotation=35, ha='right', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, max(values) * 1.25 if max(values) > 0 else 1)

fig.suptitle('Structure and fragility by mode '
             '(normalised, so modes of very different size can be compared)',
             fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGURES / 'mode_structure_comparison.png', dpi=FIG_DPI)
plt.show()

print('saved:', FIGURES / 'mode_structure_comparison.png')

## 16. איור 3 - היכן ממוקם כל אופן על המפה

תרשים פיזור גאוגרפי של כל התחנות, צבוע לפי האופן המשרת אותן. שלוש החלטות ציור חשובות:

* האופנים מצוירים **מן הגדול לקטן**, כך שהאופנים הקטנים אינם נקברים תחת עשרות אלפי נקודות אוטובוס;
* גודל הסמן משתנה ביחס הפוך לגודל האופן, מאותה סיבה;
* יחס הממדים מתוקן באמצעות `cos(latitude)` כך שהמדינה אינה נמתחת אופקית.

תחנה המשרתת כמה אופנים מצוירת פעם אחת לכל אופן, כך שתחנות רב-אופניות מופיעות כסמנים חופפים - וזה בדיוק מה שהופך את קווי הרכבת והרכבת הקלה לנראים מעל שכבת האוטובוסים. תחנות ללא קואורדינטות שמישות מוחרגות ונספרות. `MAP_MAX_POINTS_PER_MODE` מבצע דגימת-משנה לכל אופן שמעל הסף (עם seed קבוע) אך ורק כדי לשמור על קובץ PNG קטן; התא מדווח אם הדבר קרה.

In [ ]:
# --- Figure 3: stops coloured by mode ------------------------------------
map_df = stop_mode_geo.dropna(subset=['lat', 'lon']).copy()
dropped = len(stop_mode_geo) - len(map_df)
if map_df.empty:
    raise ValueError('No stop carries usable coordinates - check stops_clean.csv '
                     'from notebook 01_data_preparation.')

order = map_df.groupby('mode_label').size().sort_values(ascending=False).index.tolist()
palette = dict(zip(order, sns.color_palette('tab10', max(len(order), 3))))
rng = np.random.default_rng(SEED)

fig, ax = plt.subplots(figsize=(8, 11))
subsampled = []
for depth, mode in enumerate(order):
    sub = map_df[map_df['mode_label'] == mode]
    total = len(sub)
    if total > MAP_MAX_POINTS_PER_MODE:
        keep = rng.choice(total, MAP_MAX_POINTS_PER_MODE, replace=False)
        sub = sub.iloc[np.sort(keep)]
        subsampled.append(mode)
    size = 2 if total > 5_000 else (12 if total > 500 else 34)
    alpha = MAP_ALPHA if total > 5_000 else 0.85
    ax.scatter(sub['lon'], sub['lat'], s=size, alpha=alpha, linewidths=0,
               color=palette[mode], zorder=depth + 1,
               label=f'{mode} ({total:,} stops)')

ax.set_aspect(1 / np.cos(np.deg2rad(float(map_df['lat'].mean()))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Israeli public-transport stops by mode\n'
             '({:,} distinct stops, {} modes)'.format(map_df['stop_id'].nunique(),
                                                      len(order)))
ax.legend(loc='lower left', fontsize=8, markerscale=2.5, framealpha=0.9)
fig.tight_layout()
fig.savefig(FIGURES / 'mode_map.png', dpi=FIG_DPI)
plt.show()

print(f'(mode, stop) rows without coordinates, excluded: {dropped:,}')
if subsampled:
    print('subsampled for drawing only:', ', '.join(subsampled))
print('saved:', FIGURES / 'mode_map.png')

## 17. מדוע האופנים שונים מבנית

הטבלאות שלעיל הן ה*מה*. חלק זה הוא ה*מדוע*, משום שההבדלים המבניים אינם מקריות של הנתונים - הם נובעים מהאופן שבו כל אופן תחבורה נבנה וממומן. (מספרי המסלולים, הנסיעות והמפעילים המצוטטים כאן הם הערכים שה-feed הזה מפיק; התאים שלעיל מחשבים אותם מחדש, ולכן אם ה-feed יעודכן אי פעם, יש לסמוך על הטבלאות ולא על הפרוזה הזו.)

**אוטובוס (`route_type = 3`) - רשת ארצית צפופה.** 6,796 רשומות מסלול, 412,544 נסיעות מתוזמנות, 29 מפעילים, ולמעשה כל מצאי התחנות של המדינה. אוטובוסים נוסעים על כבישים שכבר קיימים, ולכן הוספת קו עולה לוח זמנים וכלי רכב, ולא זכות דרך. התוצאה היא גרף עם עשרות אלפי צמתים, דרגה ממוצעת מעל 3, ומסלולים חלופיים אמיתיים בליבות המטרופוליניות. אולם אותה זולות מייצרת גם הסתעפויות ארוכות, דקות וחד-קוויות אל יישובים קטנים ואזורים כפריים, וההסתעפויות הללו הן עצים - וזו הסיבה ששכבת האוטובוסים, על אף היותה האופן היתיר ביותר במדינה, עדיין תורמת אלפי נקודות חיתוך. השבירות שלה היא *פריפריאלית*: חתכים רבים, שכל אחד מהם מבודד מספר קטן של תחנות.

**רכבת (`route_type = 2`) - גרף דליל הקרוב למסלול.** 962 רשומות מסלול אך רק 1,188 נסיעות מתוזמנות וכ-67 תחנות, כולן של רכבת ישראל. מסילה יקרה והגאוגרפיה של הרכבת היא גאוגרפיית מסדרון: הרשת היא בעצם ציר צפון-דרום אחד עם מספר הסתעפויות. במונחי גרפים זהו מצב הקרוב למסלול (path), ולכן הדרגה הממוצעת נמצאת בסביבות 2, רוב תחנות הביניים הן נקודות חיתוך ורוב המקטעים הם גשרים. לרכבת יש את השלכת הכשל *לכל תחנה* הגבוהה ביותר מכל אופן ב-feed, וזו בדיוק הסיבה שמחברת 11 יכלה להרשות לעצמה לדמות בכוח גס (brute force) כל סגירה של תחנה בודדת.

**רכבת קלה / חשמלית (`route_type = 0`) - זעירה אך בתדירות גבוהה מאוד.** רק 8 רשומות מסלול, ואף על פי כן **2,890 נסיעות מתוזמנות** - יותר מפי שניים מכלל תוכנית הרכבת הכבדה - על פני קומץ תחנות, המופעלות על ידי שני מפעילים על מה ששמות המסלולים מזהים כמסדרון ירושלים (הדסה עין כרם עד נווה יעקב). מבנית זהו מסלול; תפעולית זהו אחד הדברים העמוסים ביותר ב-feed. זהו המקרה הברור ביותר שבו מדד גודל רשת ומדד נפח שירות מצביעים לכיוונים מנוגדים, וזו הסיבה ש-`trips_per_edge` נמצא בטבלת הפירוט.

**Cable tram (`route_type = 5`) - ארבע רשומות מסלול, כ-3,006 נסיעות.** שני מפעילים חיפאיים מופיעים תחת קוד זה: הפוניקולר "כרמלית" ושירות רכבל בין מרכזית המפרץ לאוניברסיטת חיפה. קומץ תחנות, מספרי נסיעות גבוהים מאוד, וטופולוגיה שהיא פשוטו כמשמעו קו. השאלה האם אלו שייכים בכלל למחקר על "רשת תחבורה ציבורית" היא שאלה של שיקול דעת; הם נשמרים משום שהשמטה שקטה של אופן גרועה מדיווח על אופן מוזר.

**Trolleybus (`route_type = 8`) - תיוג שגוי, לא אופן תחבורה.** 8 רשומות מסלול ורק 47 נסיעות. המפעילים הם חברות מוניות שירות בתל אביב, ולכן קוד זה משמש בפועל קווי *שירות* ולא טרוליבוסים. התייחסות אליו כאל אופן תחבורה נפרד תהיה שגיאה; התווית שבשימוש כאן (`trolleybus/taxi-coded`) אומרת זאת על פני כל טבלה.

**לפי דרישה (`route_type = 715`) - 14 רשומות מסלול, 458 נסיעות.** שירותים כפריים לפי דרישה המופעלים על ידי מועצות אזוריות ושני מפעילי אוטובוסים. אלו אוטובוסים בכל מובן פיזי; GTFS מפריד אותם משום שמודל השירות שלהם שונה. הם תורמים מספר קטן של שברי רשת כפריים דלילים מאוד ומנותקים ברובם.

**השורה התחתונה המבנית.** יתירות עוקבת אחר עלות התשתית - ביחס הפוך. אופנים הזקוקים לזכות דרך ייעודית (רכבת, רכבת קלה, רכבל) הם לינאריים, מינימליים ולכן פגיעים מקסימלית לחתכים, ואף על פי כן הם נושאים נתח בלתי פרופורציונלי מן השירות. האופן שאינו זקוק לתשתית ייעודית (אוטובוס) הוא היחיד בעל מסלולים חלופיים אמיתיים - והוא גם היחיד שכשליו מקומיים ולא מפצלי-רשת. מסקנת חוסן הנגזרת מן הגרף הממוזג הכולל את כל האופנים מערבבת שני משטרים שונים מאוד אלו, וזהו הנימוק המתודולוגי המרכזי לקיומה של מחברת זו.

**הסתייגויות, במפורש.**

* צלעות הן **שכנות לוח זמנים, לא שכנות מסילה**. שירות אקספרס המדלג על תחנות ביניים יוצר צלע "קיצור דרך" שאין לה מקבילה פיזית, מה שמנפח מעט את הדרגה ברכבת וברכבת הקלה.
* המשקלים הם **נסיעות מתוזמנות, לא נוסעים**. GTFS אינו נושא נתוני ביקוש, ולכן "חשוב" כאן פירושו תמיד חשוב ברשת ההיצע.
* שיוך האופן הוא כל מה שהמפעיל הצהיר עליו ב-`routes.txt`. כפי שמראה `route_type = 8`, הצהרה זו אינה תמיד בעלת משמעות: ששת ה"אופנים" הם למעשה שלוש מציאויות תחבורתיות - אוטובוסים, רכבת על מסילה קבועה, וכמה שירותי רכבל בחיפה.
* ה-feed הוא תצלום מצב של חלון לוח זמנים אחד. מספרי הנסיעות הם ספירות של רשומות נסיעה מתוזמנות בחלון זה, ולא נסיעות ליום.

## מסקנות

* **ה-feed הישראלי הוא רשת גדולה אחת בתוספת חמש קטנות.** האוטובוס מהווה את הרוב המכריע של התחנות, המקטעים והנסיעות; כל אופן אחר קטן בשני סדרי גודל לפחות בציר אחד לפחות. כל אמירה על "רשת התחבורה הישראלית" המחושבת על הגרף הממוזג היא, מספרית, אמירה על רשת האוטובוסים.
* **גודל ונפח שירות מניבים דירוגים שונים.** לרכבת הכבדה יש פי מאה ויותר רשומות מסלול מאשר לרכבת הקלה, אך פחות ממחצית מנסיעותיה המתוזמנות, ו-cable tram - ארבע רשומות מסלול - מפעיל יותר נסיעות מן הרכבת הכבדה. דירוג האופנים לפי גודל רשת ולפי עצימות שירות מפיק סדרים כמעט הפוכים, וזו הסיבה ששניהם מדווחים.
* **שבירות קשורה ביחס הפוך לעלות התשתית.** אופני המסילה הקבועה הם גרפים הקרובים למסלול: דרגה ממוצעת קרובה ל-2, רוב התחנות נקודות חיתוך, רוב המקטעים גשרים. האוטובוס הוא האופן היחיד בעל מסלולים חלופיים משמעותיים, ואף על פי כן במונחים מוחלטים הוא עדיין תורם אלפי נקודות חיתוך משום שהפריפריה שלו היא יער של הסתעפויות ללא מוצא. אלו שני משטרי כשל שונים באמת, והגרף הממוזג ממצע ביניהם.
* **שניים מששת ה"אופנים" הם ארטיפקטים של תיוג.** `route_type = 8` משמש מפעילי מוניות שירות ולא טרוליבוסים, ו-`route_type = 715` הוא שירות אוטובוס לפי דרישה. אף אחד מהם אינו רשת פיזית נפרדת. הם מדווחים משום שהשמטתם בשקט הייתה חוסר יושרה, אך שום מסקנה מבנית לא צריכה להישען עליהם - 47 נסיעות על פני 8 רשומות מסלול אינן רשת, וכל אחוז המחושב עליהן הוא רעש.
* **הפיצול מתיישב עם שלב 02.** האיחוד של קבוצות התחנות והמקטעים לפי אופן מתאים לגרף הרשת השלמה שנבנה במחברת 02, עד כדי הפערים המתועדים והצפויים (תחנות ללא מקטע הן צמתים כאן אך לא שם; מקטע המשורת על ידי שני אופנים מופיע בשני מונים). זו הראיה לכך שה-join `trip -> route -> mode` לא ניתב שירות באופן שגוי בשקט, וזו הטעות היחידה שמחברת זו הייתה עלולה לעשות באופן בלתי נראה.
* **מה זה ממסגר.** מחברת 15 לוקחת את שכבת האוטובוסים לבדה (משטר הרשת הצפופה), 16 לוקחת את הרכבת הקלה והאופנים המשניים (משטר המסלול), 17 לוקחת את התחנות שבהן האופנים נפגשים - והמספרים כאן אומרים מראש איזו מהן היא בעיית גרף גדול הדורשת קירוב ואיזו ניתנת לפתרון מדויק בכוח גס.